# 04 — Analysis & Headline Findings

**Goal:** Combine the LLM baseline (notebook 02) and the fine-tune sweep (notebook 03) into the headline figures, cost analysis, statistical-significance test, and failure-mode breakdown that anchor the README.

## Sections

1. Load all predictions
2. Headline figure: macro-F1 vs training-set size, with LLM baseline
3. Cost analysis: $/1k inferences, payback chart
4. Latency comparison
5. Statistical significance test (paired bootstrap at the candidate crossover n)
6. Failure-mode analysis: where do LLM and small model disagree?
7. Export final figures to `results/figures/`
8. Writeup notes for the README

## 1. Load predictions

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
from src.eval import load_llm_predictions, load_finetune_predictions, compute_metrics, paired_bootstrap_macro_f1

FIG_DIR = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# TODO: llm_df = load_llm_predictions(); finetune_df = load_finetune_predictions()

## 2. Headline figure — crossover plot

- X axis: training-set size (log scale)
- Y axis: macro-F1
- Solid line + 95% CI band: DistilBERT performance across 5 seeds at each n
- Horizontal dashed line: Claude Sonnet 4.6 baseline
- Annotated: the crossover point where DistilBERT meets/beats the LLM

In [ ]:
# TODO: compute per-(n, seed) macro-F1 from finetune_df; plot mean + 95% CI band across seeds at each n
# TODO: overlay LLM macro-F1 as horizontal line

## 3. Cost analysis

In [ ]:
# Posted Sonnet 4.6 pricing at time of writing (verify before publishing):
CLAUDE_INPUT_COST_PER_MTOK = 3.00   # USD
CLAUDE_OUTPUT_COST_PER_MTOK = 15.00  # USD

# Fine-tune costs: one-time training cost (Colab T4 ~= free or $10/month Colab Pro)
# + per-inference cost on a Hugging Face Inference Endpoint or self-hosted (~$0/inference once running)

# TODO: compute payback curve — at X queries/month, fine-tune amortises in Y weeks

## 4. Latency

In [ ]:
# TODO: compare median + p95 latency for both approaches

## 5. Statistical significance test — paired bootstrap

Identify the candidate crossover *n* visually from section 2 (the smallest n where mean DistilBERT macro-F1 meets/exceeds the LLM). Then run a paired bootstrap on the per-instance test predictions to test whether the gap actually crosses zero.

**Interpretation:**
- 95% CI strictly above 0: DistilBERT significantly beats the LLM at this n
- 95% CI strictly below 0: LLM still significantly better
- 95% CI crosses 0: they're statistically indistinguishable at this n — the headline becomes "matches" not "beats"

In [ ]:
CANDIDATE_CROSSOVER_N = None  # TODO: set from section 2 figure
PREFERRED_SEED = 0  # TODO: or aggregate across seeds via pooled predictions

# TODO: y_true = test true labels (3,080 rows)
# TODO: y_pred_distilbert = finetune_df predictions at (n=CANDIDATE_CROSSOVER_N, seed=PREFERRED_SEED)
# TODO: y_pred_llm = llm_df predictions
# result = paired_bootstrap_macro_f1(y_true, y_pred_distilbert, y_pred_llm, n_bootstrap=5000)
# print(result)

## 6. Failure-mode analysis

For each test example, do the LLM and the fine-tuned model agree?
- Both correct: easy case
- Both wrong: hard case (genuinely ambiguous?)
- LLM right, DistilBERT wrong: needs more training data / different architecture
- DistilBERT right, LLM wrong: where does the LLM have systematic blind spots?

Pre-commit to looking at three patterns specifically:
1. **Intent confusability pairs** — which intent pairs each model confuses most
2. **Query length effect** — are very short / very long queries where one model wins?
3. **Named-entity-heavy queries** — do queries mentioning specific cards/banks/amounts behave differently?

This is the most interesting analytical section. Spend time here.

In [ ]:
# TODO

## 7. Export final figures

In [ ]:
# TODO: save the 4-5 figures used in the README — high DPI PNG for clear LinkedIn / GitHub rendering

## 8. Writeup notes

Fill these in as the analysis surfaces them — they become the TL;DR, Findings, and Limitations sections of the README.

**Headline finding:**
> *(one sentence — fill in)*

**Crossover point:**
> *(n ≈ X examples)*

**Paired-bootstrap 95% CI on macro-F1 gap at crossover:**
> *(e.g. [-0.012, +0.018] — CI crosses zero, so the headline is "DistilBERT matches Sonnet at n≈X" not "beats")*

**Cost ratio at crossover:**
> *(small model ~Y× cheaper per inference)*

**Latency ratio:**
> *(small model ~Z× faster)*

**Failures where the LLM still wins:**
> *(describe pattern)*

**Surprises:**
> *(anything counter-intuitive — these are gold for the writeup)*